# 讓 Agent 接得住前後文

這份教材示範如何用同一個對話識別碼連續跑多輪，讓 Agent 保存前面聊過的內容。

In [ ]:
from pathlib import Path
import os, sys, subprocess

if not Path('agentic_sdk').exists():
    if not Path('Agentic-SDK').exists():
        subprocess.run(['git', 'clone', 'https://github.com/R300-AI/Agentic-SDK.git'], check=True)
    os.chdir('Agentic-SDK')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

In [ ]:
from agentic_sdk import Workflow
from agentic_sdk.modules import DirectAnswerAction, KeywordRetrieve, PassThroughPerceive

workflow = Workflow(
    workflow_name='多輪問答 Agent',
    perceive=PassThroughPerceive(),
    retrieve=KeywordRetrieve(items=[
        {'keywords': ['專案代號', 'aurora'], 'content': '使用者提到的專案代號是 Aurora。'},
        {'keywords': ['會議', '明天'], 'content': '明天會議需要準備專案摘要。'},
    ]),
    action=DirectAnswerAction(),
)

session_id = 'demo-user-001'
first = workflow.run('請記住，這次專案代號是 Aurora。', session_id=session_id)
second = workflow.run('那明天會議我要準備什麼？', session_id=session_id)

print('第一輪:', first.final_message)
print('第二輪:', second.final_message)

## 查看對話紀錄

同一個 `session_id` 會讓 workflow 保存前面幾輪 user / assistant 的順序。

In [ ]:
memory = second.memory
for index, turn in enumerate(memory.turns, start=1):
    print(index, turn.role, '=>', turn.content)

## 觀察重點

本輪狀態負責保存這一次執行的中間資料；記憶類型負責保存同一段對話的前後文。兩者不要混在一起看。